In [4]:
import pandas as pd
import numpy as np
import re

In [5]:
medical_term_mapping = {
    'ABSCESS': r'.*ABSCESS.*',
    'ANKYLOSIS': r'.*ANKYLOSIS.*',
    'ANEMIA': r'.*ANEMIA.*',
    'APHONIA': r'.*APHONIA.*',
    'ASCITES': r'.*ASCITES.*',
    'BILIOUS': r'.*BILIOUS.*',
    'BOIL': r'.*BOIL.*',
    'BRONCHITIS': r'.*BRONCHITIS.*',
    'BRUISE': r'.*BRUISE.*',
    'BUNION': r'.*BUNION.*',
    'CATARRH': r'.*CATARRH.*',
    'CONJUNCTIVITIS': r'.*CONJUNCTIVITIS.*',
    'CONSTIPATION': r'.*CONSTIPATION.*',
    'CONVALESCENT': r'.*CONVAL.*',
    'COUGH': r'.*COUGH.*',
    'DEAFNESS/DEAF': r'.*DEAF.*',
    'DEBILITY': r'.*DEBILITY.*',
    'DIARRHEA': r'.*DIARRHEA.*',
    'DIPHTHERIA': r'.*DIPHTHERIA.*',
    'DYSENTERY': r'.*DYSENTERY.*',
    'EDEMA': r'.*EDEMA.*',
    'ENDOCARDITIS': r'.*ENDOCARDITIS.*',
    'EPILEPTIC': r'.*EPILEPT.*',
    'ERYSIPELAS': r'.*ERYSIPELAS.*',
    'EYES': r'.*EYES.*',
    'FEVER': r'.*FEVER.*',
    'GASTROINTESTIN': r'.*GASTRO.*',
    'GONORRHEA': r'.*GONORRHEA.*',
    'GSW': r'.*GSW.*',
    'HEART': r'.*HEART.*',
    'HEMORRHOIDS': r'.*HEMORRHOI.*',
    'HERNIA': r'.*HERNIA.*',
    'HYPERTROPHY': r'.*HYPERTRO.*',
    'ICTEROID': r'.*ICTEROID.*',
    'INFLAMMATION': r'.*INFLAM.*',
    'JAUNDICE': r'.*JAUNDICE.*',
    'LARYNGITIS': r'.*LARYNGITIS.*',
    'LUMBAGO': r'.*LUMBAGO.*',
    'LUNG': r'.*LUNG.*',
    'MALARIA': r'.*MALARIA.*',
    'MENINGITIS': r'.*MENINGITIS.*',
    'MUMPS': r'.*MUMPS.*',
    'NEPHRITIS': r'.*NEPHRITIS.*',
    'NERVES/NERVOUS': r'.*NERV.*',
    'PAROTITIS': r'.*PAROTITIS.*',
    'PLEURITIC/PLEURI': r'.*PLEURI.*',
    'PNEUMONIA': r'.*PNEUMONIA.*',
    'PURULENT': r'.*PURULENT.*',
    'RHEUMATISM': r'.*RHEUMATISM.*',
    'SCURVY': r'.*SCURVY.*',
    'SICK': r'.*SICK.*',
    'SMALLPOX': r'.*SMALLPOX.*',
    'SPRAIN': r'.*SPRAIN.*',
    'SYPHILIS': r'.*SYPHILIS.*',
    'THYROID': r'.*THYROID.*',
    'TYPHOID': r'.*TYPHOID.*',
    'TONSIL': r'.*TONSIL.*',
    'TUBERCULOSIS/TUBERCULAR': r'.*TUBERCUL.*',
    'ULCER': r'.*ULCER.*',
    'ULCERATION': r'.*ULCERATION.*',
    'VARIOLOID': r'.*VARIOLOID.*',
    'VULNUS SCLOPIT/VS': r'.*VULNUS.*',
    'WND/WOUND': r'.*WND.*',
    'MEASLES': r'.*MEASLES.*',
    'RUBEOLA': r'.*RUBEOLA.*'
}



In [47]:
#commented out conditions didn't appear in the group of 351 and/or didn't have an assigned condition group or stand alone status 
condition_groups = {
    'ABSCESS': 'skin/surface infection',
    'ANKYLOSIS': 'joint',
    'ANEMIA': 'cardiovascular',
    'APHONIA': 'pulmonary',
    #'ASCITES': r'.*ASCITES.*',
    'BILIOUS': 'gastrointestinal',
    'BOIL': 'skin/surface infection',
    'BRONCHITIS': 'pulmonary',
    'BRUISE': 'mechanical',
    'BUNION': 'mechanical',
    'CATARRH': 'pulmonary',
    'CONJUNCTIVITIS': 'ocular',
    'CONSTIPATION': 'gastrointestinal',
    'CONVALESCENT': 'malaise',
    'COUGH': 'pulmonary',
    'DEAFNESS/DEAF': 'otic',
    'DEBILITY': 'malaise',
    'DIARRHEA': 'gastrointestinal',
    'DIPHTHERIA': 'infectious',
    'DYSENTERY': 'gastrointestinal',
    'EDEMA': 'cardiovascular',
    'ENDOCARDITIS': 'cardiovascular',
    'EPILEPTIC': 'nervous',
    'ERYSIPELAS': 'skin/surface infection',
    'EYES': 'ocular',
    'FEVER': 'fever',
    'GASTROINTESTIN': 'gastrointestinal',
    'GONORRHEA': 'venereal',
    'GSW': 'wound',
    'HEART': 'cardiovascular',
    'HEMORRHOIDS': 'gastrointestinal',
    'HERNIA': 'mechanical',
    #'HYPERTROPHY': r'.*HYPERTRO.*',
    'ICTEROID': 'infectious',
    'INFLAMMATION': 'inflammation',
    'JAUNDICE': 'infectious',
    'LARYNGITIS': 'pulmonary',
    'LUMBAGO': 'mechanical',
    'LUNG': 'pulmonary',
    'MALARIA': 'malaria',
    'MENINGITIS': 'infectious',
    'MUMPS': 'infectious',
    'NEPHRITIS': 'nephritis',
    'NERVES/NERVOUS': 'nervous',
    'PAROTITIS': 'infectious',
    'PLEURITIC/PLEURI': 'pulmonary',
    'PNEUMONIA': 'pulmonary',
    'PURULENT': 'skin/surface infection',
    'RHEUMATISM': 'rheumatism',
    'SCURVY': 'scurvy',
    'SICK': 'sick',
    'SMALLPOX': 'smallpox',
    'SPRAIN': 'mechanical',
    'SYPHILIS': 'venereal',
    #'THYROID': r'.*THYROID.*',
    'TYPHOID': 'gastrointestinal',
    'TONSIL': 'pulmonary',
    'TUBERCULOSIS/TUBERCULAR': 'tuberculosis',
    'ULCER': 'skin/surface infection',
    'ULCERATION': 'skin/surface infection',
    'VARIOLOID': 'smallpox',
    'VULNUS SCLOPIT/VS': 'wound',
    'WND/WOUND': 'wound',
    'MEASLES': 'measles',
    'RUBEOLA': 'measles'
}

In [48]:
def standardize_terms(text, term_mapping=medical_term_mapping, exclude_no_match=True):
    if pd.isna(text):
        return text
    
    # Check each pattern and return the standardized term if found
    for standard_term, pattern in term_mapping.items():
        if re.search(pattern, text, re.IGNORECASE):
            return standard_term
    
    if exclude_no_match:
        return np.nan #return nan if text doesn't contain a search term
    else:
        return text  #return original if no match found


In [49]:
data = pd.read_csv('~/projects/measles_analysis/measles_set_clean.csv', index_col='recidnum', na_values='0')
data.shape

(351, 15)

In [50]:
print("Number of cells in the dataframe that identify a condition (are not empty):", pd.notna(data.values.ravel()).sum())
print("Number of conditions contained in data before standardizing terms:", len(pd.unique(data.values.ravel())))

Number of cells in the dataframe that identify a condition (are not empty): 960
Number of conditions contained in data before standardizing terms: 226


In [51]:
#standardize terms in each column of data
for col in data.columns:
    data[col] = (data[col].apply(standardize_terms))

In [52]:
print("Number of conditions contained in data after standardizing terms:", len(pd.unique(data.values.ravel())))

Number of conditions contained in data after standardizing terms: 46


In [53]:
conditionGroupData = data.replace(condition_groups)
conditionGroupData.shape

(351, 15)

In [54]:
print("Number of condition groups contained in data after standardizing terms:", len(pd.unique(conditionGroupData.values.ravel())))

Number of condition groups contained in data after standardizing terms: 22


In [55]:
data.to_csv('~/projects/measles_analysis/cleanedConditionData.csv')
conditionGroupData.to_csv('~/projects/measles_analysis/conditionGroupData.csv')

In [56]:
n_bootstraps = 10000
samples = []
counts = []

for _ in range(n_bootstraps):
    sample = conditionGroupData.sample(n=len(conditionGroupData), replace=True)
    samples.append(sample)

    counts.append((pd.Series(sample.values.ravel())).value_counts())

In [40]:
test = pd.Series(samples[0].values.ravel())

In [62]:
test = pd.DataFrame(counts)

In [64]:
test.columns

Index(['measles', 'sick', 'gastrointestinal', 'pulmonary', 'fever', 'wound',
       'malaise', 'inflammation', 'smallpox', 'rheumatism', 'ocular',
       'nephritis', 'infectious', 'mechanical', 'cardiovascular', 'otic',
       'tuberculosis', 'venereal', 'malaria', 'skin/surface infection',
       'nervous'],
      dtype='object')